# Microplastic Detection — YOLOv8m Multi-Class Training (Kaggle)

> **3-class detection: fiber, film, fragment**

## Kaggle Setup

1. **Upload dataset** as Kaggle Dataset: upload `yolo_augmented_balanced/` folder
   - Name it e.g. `mp-yolo-augmented-balanced`
   - Must contain: `dataset.yaml`, `images/{train,val}/`, `labels/{train,val}/`

2. **Add dataset** to this notebook via sidebar → Add Data

3. **Enable GPU** (Settings → Accelerator → GPU T4 x2)

4. **Enable Internet** (Settings → Internet → On)

## Where Models Are Saved

```
/kaggle/working/
├── best.pt                              ← easy download
└── experiments/yolo/
    └── mp_yolov8m_1024_adamw/
        ├── weights/best.pt              ← best model
        ├── weights/last.pt              ← last checkpoint
        ├── results.png                  ← training curves
        └── confusion_matrix_normalized.png
```

After training: **Save Version** → download from Output tab.

## 1. Environment Setup

In [ ]:
!pip install ultralytics>=8.1.0 --quiet

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

## 2. GPU Verification & Reproducibility

In [ ]:
import torch, random, numpy as np, os, psutil

assert torch.cuda.is_available(), "No GPU — enable in Settings > Accelerator"

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"RAM: {psutil.virtual_memory().total/1e9:.1f} GB")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
print(f"Seed: {SEED}")

## 3. Dataset Setup

**Change `KAGGLE_DATASET_NAME`** below to match your uploaded dataset.

In [ ]:
import os, shutil, yaml
from pathlib import Path

# >>> CHANGE THIS <<<
KAGGLE_DATASET_NAME = "mp-yolo-augmented-balanced"

INPUT_PATH  = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
LOCAL_DATASET_PATH = "/kaggle/working/dataset"
OUTPUT_PATH = "/kaggle/working/experiments/yolo"
os.makedirs(OUTPUT_PATH, exist_ok=True)

assert os.path.exists(INPUT_PATH), (
    f"Dataset not found at {INPUT_PATH}\n"
    f"Available: {os.listdir('/kaggle/input/')}")

# Auto-detect dataset.yaml location
if os.path.exists(os.path.join(INPUT_PATH, 'dataset.yaml')):
    SRC_PATH = INPUT_PATH
else:
    found = list(Path(INPUT_PATH).rglob('dataset.yaml'))
    assert found, f"No dataset.yaml found under {INPUT_PATH}"
    SRC_PATH = str(found[0].parent)

if not os.path.exists(LOCAL_DATASET_PATH):
    print("Copying dataset to working directory...")
    shutil.copytree(SRC_PATH, LOCAL_DATASET_PATH)
    print("Done.")

DATASET_PATH = LOCAL_DATASET_PATH
YAML_PATH = f"{DATASET_PATH}/dataset.yaml"

with open(YAML_PATH) as f:
    ds_config = yaml.safe_load(f)
ds_config['path'] = DATASET_PATH
with open(YAML_PATH, 'w') as f:
    yaml.dump(ds_config, f, default_flow_style=False)

print(yaml.dump(ds_config, default_flow_style=False))

## 3.1 Dataset Verification

In [ ]:
from collections import Counter
import numpy as np

CLASS_NAMES = {0: "fiber", 1: "film", 2: "fragment"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for split in ["train", "val"]:
    img_dir = Path(DATASET_PATH) / "images" / split
    lbl_dir = Path(DATASET_PATH) / "labels" / split
    images = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    labels = {f.stem for f in lbl_dir.iterdir() if f.suffix == ".txt"}

    class_counts = Counter()
    bbox_widths, bbox_heights = [], []
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                class_counts[int(float(parts[0]))] += 1
                bbox_widths.append(float(parts[3]))
                bbox_heights.append(float(parts[4]))

    total = sum(class_counts.values())
    print(f"\n[{split.upper()}] images: {len(images)}, labels: {len(labels)}, objects: {total}")
    for cls_id in sorted(class_counts):
        name = CLASS_NAMES.get(cls_id, f"class_{cls_id}")
        print(f"  {name:12s}: {class_counts[cls_id]:5d} ({100*class_counts[cls_id]/total:.1f}%)")

    if bbox_widths:
        w, h = np.array(bbox_widths), np.array(bbox_heights)
        print(f"  Bbox (px@1024): median {np.median(w)*1024:.0f}x{np.median(h)*1024:.0f}")

## 4. Model Training

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil, datetime

MODEL      = "yolov8m.pt"
IMGSZ      = 1024
BATCH_SIZE = 4
EPOCHS     = 200
PATIENCE   = 50
EXPERIMENT = "mp_yolov8m_1024_adamw"

# ---------------------------------------------------------------------------
# Auto-backup callback — copies weights every 50 epochs
# ---------------------------------------------------------------------------
BACKUP_EVERY = 50
BACKUP_ROOT  = Path("/kaggle/working/backups")

def auto_backup(trainer):
    epoch = trainer.epoch + 1
    if epoch % BACKUP_EVERY != 0:
        return
    weights_dir = Path(trainer.save_dir) / "weights"
    backup_dir  = BACKUP_ROOT / f"epoch_{epoch:04d}"
    backup_dir.mkdir(parents=True, exist_ok=True)
    for name in ("best.pt", "last.pt"):
        src = weights_dir / name
        if src.exists():
            shutil.copy2(src, backup_dir / name)
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"\n[AUTO-BACKUP] epoch {epoch} → {backup_dir}  ({ts})\n")

model = YOLO(MODEL)
model.add_callback("on_train_epoch_end", auto_backup)

print(f"Training: {MODEL} @ {IMGSZ}, batch={BATCH_SIZE}, epochs={EPOCHS}")
print(f"Auto-backup: every {BACKUP_EVERY} epochs → {BACKUP_ROOT}")

results = model.train(
    data=YAML_PATH,
    epochs=EPOCHS, batch=BATCH_SIZE, imgsz=IMGSZ,
    device=0, workers=2, seed=SEED, deterministic=True,

    optimizer="AdamW", lr0=1e-3, lrf=0.01,
    momentum=0.937, weight_decay=5e-4,
    warmup_epochs=5, warmup_momentum=0.8, warmup_bias_lr=0.1,
    cos_lr=True,

    patience=PATIENCE,

    augment=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15, translate=0.15, scale=0.5,
    shear=5.0, perspective=0.0005,
    flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.3,
    close_mosaic=30,

    box=7.5, cls=0.5, dfl=1.5,
    label_smoothing=0.05,

    amp=True,
    cache=False, rect=False, multi_scale=False,

    project=OUTPUT_PATH, name=EXPERIMENT, exist_ok=True,
    save=True, save_period=25,
    verbose=True, plots=True,
)

print(f"\nBest weights: {OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt")


## 5. Validation & Metrics

In [ ]:
# ==============================================================================
# 5. VALIDATION (self-contained — can run after kernel restart)
# ==============================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics>=8.1.0", "-q"], check=True)

from ultralytics import YOLO
from pathlib import Path

OUTPUT_PATH  = "/kaggle/working/experiments/yolo"
EXPERIMENT   = "mp_yolov8m_1024_adamw"
DATASET_PATH = "/kaggle/working/dataset"
YAML_PATH    = f"{DATASET_PATH}/dataset.yaml"
IMGSZ        = 1024
BATCH_SIZE   = 4
BACKUP_ROOT  = Path("/kaggle/working/backups")

def find_best_weights(primary, backup_root):
    if Path(primary).exists():
        return primary, "primary path"
    backups = sorted(backup_root.glob("epoch_*/best.pt")) if backup_root.exists() else []
    if backups:
        return str(backups[-1]), f"backup ({backups[-1].parent.name})"
    candidates = sorted(Path("/kaggle/working").rglob("best.pt"))
    if candidates:
        return str(candidates[-1]), "broad search"
    raise FileNotFoundError(
        "best.pt not found. /kaggle/working/ is wiped on kernel restart.\n"
        "Click 'Save Version' after training, then attach saved output as input data."
    )

BEST_WEIGHTS, source = find_best_weights(
    f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt", BACKUP_ROOT
)
print(f"[OK] Using best.pt from {source}: {BEST_WEIGHTS}")

model = YOLO(BEST_WEIGHTS)

metrics = model.val(
    data=YAML_PATH, imgsz=IMGSZ, batch=BATCH_SIZE,
    conf=0.001, iou=0.6, max_det=1000,
    plots=True, save_json=True,
)

print(f"\nmAP@0.50      : {metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95 : {metrics.box.map:.4f}")
print(f"Precision     : {metrics.box.mp:.4f}")
print(f"Recall        : {metrics.box.mr:.4f}")

for i, name in enumerate(["fiber", "film", "fragment"]):
    if i < len(metrics.box.ap50):
        print(f"  {name:12s}: AP50={metrics.box.ap50[i]:.4f}")


## 5.1 Training Curves & Predictions

In [ ]:
# ==============================================================================
# 5.1 TRAINING CURVES & PREDICTIONS (self-contained)
# ==============================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics>=8.1.0", "-q"], check=True)

import matplotlib.pyplot as plt, cv2
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

OUTPUT_PATH  = "/kaggle/working/experiments/yolo"
EXPERIMENT   = "mp_yolov8m_1024_adamw"
DATASET_PATH = "/kaggle/working/dataset"
IMGSZ        = 1024
BACKUP_ROOT  = Path("/kaggle/working/backups")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_best_weights(primary, backup_root):
    if Path(primary).exists():
        return str(primary)
    backups = sorted(backup_root.glob("epoch_*/best.pt")) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path("/kaggle/working").rglob("best.pt"))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError("best.pt not found. Did training complete?")

BEST_WEIGHTS = find_best_weights(
    f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt", BACKUP_ROOT
)
print(f"Using: {BEST_WEIGHTS}")

results_dir = Path(BEST_WEIGHTS).parent.parent

for pf in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"]:
    if (results_dir / pf).exists():
        print(f"\n--- {pf} ---")
        display(Image(filename=str(results_dir / pf), width=800))

# Filter to image files only — skip .npy and other non-image formats
val_images_dir = Path(DATASET_PATH) / "images" / "val"
sample_images = sorted(
    p for p in val_images_dir.glob("*") if p.suffix.lower() in IMG_EXTS
)[:6]

if sample_images:
    model = YOLO(BEST_WEIGHTS)
    preds = model([str(p) for p in sample_images], imgsz=IMGSZ, conf=0.25, iou=0.45)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, r in zip(axes.flatten(), preds):
        ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f"{len(r.boxes)} detections"); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print(f"[WARN] No image files found in {val_images_dir}")


## 6. Export & Save

In [ ]:
# ==============================================================================
# 6. EXPORT & SAVE (self-contained)
# ==============================================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics>=8.1.0", "-q"], check=True)

from ultralytics import YOLO
from pathlib import Path
import shutil

OUTPUT_PATH = "/kaggle/working/experiments/yolo"
EXPERIMENT  = "mp_yolov8m_1024_adamw"
IMGSZ       = 1024
BACKUP_ROOT = Path("/kaggle/working/backups")

def find_best_weights(primary, backup_root):
    if Path(primary).exists():
        return str(primary)
    backups = sorted(backup_root.glob("epoch_*/best.pt")) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path("/kaggle/working").rglob("best.pt"))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError("best.pt not found. Did training complete?")

BEST_WEIGHTS = find_best_weights(
    f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt", BACKUP_ROOT
)
print(f"Using: {BEST_WEIGHTS}")

model = YOLO(BEST_WEIGHTS)
onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
print(f"ONNX: {onnx_path}")

shutil.copy2(BEST_WEIGHTS, "/kaggle/working/best.pt")
print("\nCopied best.pt to /kaggle/working/best.pt")
print("Save Version → download from Output tab")
